In [2]:
# Authenticate to Google Earth Engine, if you haven't done so already on your machine!
import ee

ee.Authenticate()


Successfully saved authorization token.


In [1]:
# Initialize Google Earth Engine API.
import ee
ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

# Basic cloud masking algorithm.
def prep_sr_l8(image):
    # Bit 0 - Fill
    # Bit 1 - Dilated Cloud
    # Bit 2 - Cirrus
    # Bit 3 - Cloud
    # Bit 4 - Cloud Shadow
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)

    # Replace the original bands with the scaled ones and apply the masks.
    return (image.addBands(optical_bands, None, True)
                 .addBands(thermal_bands, None, True)
                 .updateMask(qa_mask)
                 .updateMask(saturation_mask))

Plumas_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/Plumas_National_forest")

def create_monthly_composite(month):
    # Determine the start and end dates for each month
    start_date = ee.Date.fromYMD(2018, month, 1)
    end_date = start_date.advance(1, 'month')
    
    # Filter by date, apply preprocessing, select bands, and compute median
    composite = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
                 .filterDate(start_date, end_date)
                 .map(prep_sr_l8)
                 .select(['SR_B4', 'SR_B5'])
                 .median()
                 .set({
                     "system:time_start": start_date.millis(),
                     "month": month,
                     "year": 2018
                 }))
    return composite

# Create an Earth Engine list from 1 to 12
months = ee.List.sequence(1, 12)

# Map the function over the list of months to create a list of images
monthly_images = months.map(create_monthly_composite)

# Cast the resulting list of images back into an ImageCollection
ic = ee.ImageCollection.fromImages(monthly_images)

In [3]:
from robustraster import run

# 1. Define your custom user function
def compute_ndvi(df):
    df["ndvi"] = (df["SR_B5"] - df["SR_B4"]) / (df["SR_B5"] + df["SR_B4"])
    return df["ndvi"]

# 3. Define a small Area of Interest (AOI) as a GEE Polygon geometry.
#    The coordinates describe a bounding box located in California's
#    Sierra Nevada region, specified in WGS84 (EPSG:4326) longitude/latitude.
#    The polygon is closed by repeating the first coordinate at the end.
aoi = ee.Geometry.Polygon(
    [[[-120.5, 38.5],     # Southwest corner (lower-left)
      [-120.5, 38.55],    # Northwest corner (upper-left)
      [-120.4, 38.55],    # Northeast corner (upper-right)
      [-120.4, 38.5],     # Southeast corner (lower-right)
      [-120.5, 38.5]]]    # Close the polygon at the starting vertex
)

# 2. Run the function
run(
    # The dataset to process (e.g., an Earth Engine ImageCollection)
    dataset=ic, 
    
    # Dataset configurations
    dataset_config={
        'geometry': aoi,
        'crs': 'EPSG:3310',
        'scale': 30,
    },
    
    # Your custom function
    user_function_config={
        "user_function": compute_ndvi,
    },
    
    # Export Configuration (Automated GEE Upload!)
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "output_tiles",
        
        # Google Cloud Storage Settings
        "export_to_gcs": True,
        "gcs_bucket": "rrbucket2345",
        "gcs_folder": "ndvi-2018-exports", # Optional subfolder
        "gcs_project": "project-459a1b61-31e0-47b6-aa9", # <--- Add this!
        
        # Google Earth Engine Upload Settings
        "upload_results_to_gee": True,
        "gee_asset_path": "projects/project-459a1b61-31e0-47b6-aa9/assets/aoi/ndvi"
    }
)


Dask dashboard is available at: http://127.0.0.1:8787/status
[robustraster] Dask cluster started: <Client: 'tcp://127.0.0.1:55359' processes=16 threads=16, memory=29.59 GiB>
[robustraster] Dask workers authenticated to Earth Engine.
[robustraster] AOI tiling enabled. Streaming tiles in batches...
[robustraster] Processing tile 1 of 1
[robustraster] Running user function...


c:\anaconda3\envs\robustraster\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Bucket rrbucket2345 already exists.
Created folder: ndvi-2018-exports
Exported to GCS: gcs://rrbucket2345/ndvi-2018-exports/x-43541_-34841_y53893_59383_tile_1__time_20180101T000000.tif with bands [np.str_('ndvi')]
Exported to GCS: gcs://rrbucket2345/ndvi-2018-exports/x-43541_-34841_y53893_59383_tile_1__time_20180201T000000.tif with bands [np.str_('ndvi')]
Exported to GCS: gcs://rrbucket2345/ndvi-2018-exports/x-43541_-34841_y53893_59383_tile_1__time_20180301T000000.tif with bands [np.str_('ndvi')]
Exported to GCS: gcs://rrbucket2345/ndvi-2018-exports/x-43541_-34841_y53893_59383_tile_1__time_20180401T000000.tif with bands [np.str_('ndvi')]
Exported to GCS: gcs://rrbucket2345/ndvi-2018-exports/x-43541_-34841_y53893_59383_tile_1__time_20180501T000000.tif with bands [np.str_('ndvi')]
Exported to GCS: gcs://rrbucket2345/ndvi-2018-exports/x-43541_-34841_y53893_59383_tile_1__time_20180601T000000.tif with bands [np.str_('ndvi')]
Exported to GCS: gcs://rrbucket2345/ndvi-2018-exports/x-43541_-348

c:\anaconda3\envs\robustraster\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


[robustraster] Found 12 files across 12 time steps. Starting GEE ingestion...
[robustraster] Asset 'projects/project-459a1b61-31e0-47b6-aa9/assets/aoi' may not exist. Attempting to create it as FOLDER...
[robustraster] ✅ Successfully created FOLDER: projects/project-459a1b61-31e0-47b6-aa9/assets/aoi
[robustraster] Asset 'projects/project-459a1b61-31e0-47b6-aa9/assets/aoi/ndvi' may not exist. Attempting to create it as IMAGE_COLLECTION...
[robustraster] ✅ Successfully created IMAGE_COLLECTION: projects/project-459a1b61-31e0-47b6-aa9/assets/aoi/ndvi
[robustraster] ✅ Earth Engine task 077459df-d13c-4a16-8d5c-b45703f6d179 launched for asset projects/project-459a1b61-31e0-47b6-aa9/assets/aoi/ndvi/2018_01_01. Tiles: 1.
[robustraster] ✅ Earth Engine task 7bd7c179-b6fc-420e-9ebe-a2429c71b0ca launched for asset projects/project-459a1b61-31e0-47b6-aa9/assets/aoi/ndvi/2018_02_01. Tiles: 1.
[robustraster] ✅ Earth Engine task 5d25336b-6da5-4021-bf90-6bbea4c26fa3 launched for asset projects/project-